# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis (EDA)
### Group ID: `2026-Y2-S1-MET-23`
### Member 5: Sameeha M.S.F. (IT25103066)
### Assigned Technique: Handling Severe Class Imbalance in Emotion Labels

---
### 1. Technique Overview & Academic Justification
Our assigned dataset exhibits **severe class imbalance**:
* `sadness`: 17,339 instances (37.5% of dataset)
* `joy`: 7,861 instances (17.0%)
* `anticipation`: 7,336 instances (15.9%)
* `optimism`: 4,812 instances (10.4%)
* `anger`: 3,638 instances (7.9%)
* `fear`: 3,460 instances (7.5%)
* `disgust`: 1,670 instances (3.6%)
* `surprise`: **57 instances (0.1% — critical minority)**

**Consequences of Class Imbalance:**
1. **Accuracy Paradox:** A naive model predicting `sadness` 100% of the time achieves ~37.5% accuracy while completely failing on minority classes.
2. **Mitigation Approaches:**
   - **Cost-Sensitive Learning (Balanced Class Weights):** Penalizes errors on minority classes inversely proportional to class frequencies:
     $$w_j = \frac{N}{K \times n_j}$$
   - **Stratified K-Fold Cross-Validation:** Guarantees that every fold preserves class proportions, vital for classes like `surprise` with only 57 examples.
   - **Resampling:** Random over-sampling or SMOTE synthesis.

**Viva Objective:** Quantify imbalance ratios, explain cost-sensitive weighting formula, and demonstrate class distributions before and after balanced adjustment.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

os.makedirs('../results/eda_visualizations', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


### 2. Loading Dataset & Analyzing Class Distribution

In [ ]:
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

emotion_counts = df['emotion'].value_counts()
emotion_pct = (df['emotion'].value_counts(normalize=True) * 100).round(2)

imbalance_df = pd.DataFrame({
    'Emotion': emotion_counts.index,
    'Count': emotion_counts.values,
    'Percentage (%)': emotion_pct.values,
    'Imbalance Ratio (vs Majority)': (emotion_counts.max() / emotion_counts.values).round(1)
})
imbalance_df


### 3. Computing Balanced Class Weights for Cost-Sensitive Learning

In [ ]:
classes = np.array(np.unique(df['emotion']), dtype=str)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['emotion'].to_numpy())
class_weight_dict = dict(zip(classes, weights))

weight_df = pd.DataFrame({
    'Emotion': list(class_weight_dict.keys()),
    'Balanced Weight': [round(w, 3) for w in class_weight_dict.values()]
}).sort_values(by='Balanced Weight', ascending=False)

print("Computed Balanced Class Weights:")
weight_df


### 4. Simulating Balanced Re-Sampling Distribution

In [ ]:
# Demonstration of balanced resampling distribution for model training
# Setting an effective target cap per class for visual comparison
simulated_balanced_counts = {
    c: int(emotion_counts.mean()) if emotion_counts[c] < emotion_counts.mean() else emotion_counts[c]
    for c in classes
}
sim_df = pd.DataFrame({
    'Emotion': list(simulated_balanced_counts.keys()),
    'Rebalanced Count': list(simulated_balanced_counts.values())
})


### 5. Individual EDA Visualizations (Viva Presentation Requirement)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left Plot: Raw Emotion Imbalance Bar Chart
palette = sns.color_palette("rocket", len(emotion_counts))
bars = axes[0].bar(imbalance_df['Emotion'], imbalance_df['Count'], color=palette)
axes[0].set_title('Raw Emotion Class Distribution (Severe Imbalance)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Emotion Category', fontsize=11)
axes[0].set_ylabel('Number of Samples', fontsize=11)
axes[0].tick_params(axis='x', rotation=35)

# Annotate bars with exact count and percentage
for bar, count, pct in zip(bars, imbalance_df['Count'], imbalance_df['Percentage (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f"{pct}%\n({count:,})",
                 ha='center', va='bottom', fontsize=8, fontweight='bold')

# Right Plot: Computed Inverse Class Weights
weight_df_sorted = weight_df.sort_values(by='Balanced Weight', ascending=True)
axes[1].barh(weight_df_sorted['Emotion'], weight_df_sorted['Balanced Weight'], color='#3498db')
axes[1].set_title('Computed Cost-Sensitive Class Penalty Weights', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Inverse Frequency Weight Multiplier', fontsize=11)
axes[1].set_ylabel('Emotion Category', fontsize=11)

for i, v in enumerate(weight_df_sorted['Balanced Weight']):
    axes[1].text(v + 0.5, i, f"{v:.2f}x", va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
output_plot_path = '../results/eda_visualizations/member5_class_imbalance_distribution.png'
plt.savefig(output_plot_path, dpi=300, bbox_inches='tight')
print(f"EDA plot saved successfully to: {output_plot_path}")
plt.show()


### 6. Key Findings & Viva Talking Points (For Sameeha M.S.F.)

> **Viva Preparation Notes:**
> 1. **Why is standard accuracy invalid for this dataset?**
>    Because `sadness` accounts for 37.5% while `surprise` accounts for only 0.12% (17,339 vs. 57 samples; an imbalance ratio of ~304:1). A dummy model predicting `sadness` gets 37.5% accuracy but is practically useless. Macro-averaged F1-Score must be used.
> 2. **How do balanced class weights resolve this issue?**
>    By assigning misclassification loss inversely proportional to frequency, mistakes on `surprise` are penalized **101 times heavier** than mistakes on `sadness`, forcing model boundaries to respect minority classes.
> 3. **Why is Stratified K-Fold splitting mandatory?**
>    With only 57 `surprise` samples across 46,000+ records, random train-test splitting risks omitting `surprise` entirely from a fold. Stratified splitting preserves identical class proportions across all validation splits.
